In [ ]:
%pip install -q ultralytics pyyaml

from google.colab import drive
from pathlib import Path
from datetime import datetime
from ultralytics import YOLO

import torch
import yaml
import zipfile
import shutil
import random

drive.mount("/content/drive")

DEVICE = "cpu"
print("Обучаем на CPU")

CLASS_NAMES = {
    0: "box_hitech",
    1: "box_hitech_defect",
}

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

In [ ]:
ARCHIVE = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "yolo_detecting_boxes.zip"
)

assert ARCHIVE.is_file(), f"Не найден архив: {ARCHIVE}"

EXTRACT_DIR = Path(
    "/content/yolo_boxes_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)

with zipfile.ZipFile(ARCHIVE) as z:
    for item in z.infolist():
        target = (EXTRACT_DIR / item.filename).resolve()

        if not target.is_relative_to(EXTRACT_DIR.resolve()):
            raise ValueError(f"Недопустимый путь: {item.filename}")

    z.extractall(EXTRACT_DIR)

roots = [
    p.parent
    for p in EXTRACT_DIR.rglob("images")
    if p.is_dir() and (p.parent / "labels").is_dir()
]

assert len(roots) == 1, (
    f"Ожидалась одна папка датасета, найдены: {roots}"
)

SOURCE_ROOT = roots[0]

print("Датасет распакован:", SOURCE_ROOT)

classes_file = SOURCE_ROOT / "classes.txt"

if classes_file.is_file():
    exported_classes = [
        name.strip()
        for name in classes_file.read_text(
            encoding="utf-8-sig"
        ).splitlines()
    ]

    print("Классы экспорта:", exported_classes)

    assert exported_classes == list(CLASS_NAMES.values()), (
        "Проверьте классы в экспорте: "
        "0 — box_hitech, 1 — box_hitech_defect. "
        "Изменение YAML не меняет номера в разметке."
    )
else:
    print(
        "classes.txt отсутствует. Проверьте, что в разметке "
        "0 — box_hitech, 1 — box_hitech_defect."
    )

In [ ]:
ALLOW_RANDOM_SPLIT = False

if (
    (SOURCE_ROOT / "images/train").is_dir()
    and (SOURCE_ROOT / "images/val").is_dir()
):
    ROOT = SOURCE_ROOT
    print("Используем существующее разделение:", ROOT)

else:
    assert ALLOW_RANDOM_SPLIT, (
        "Нет папок images/train и images/val. "
        "Для независимых фотографий установите "
        "ALLOW_RANDOM_SPLIT = True. "
        "Для кадров видео сначала разделите данные по полётам."
    )

    images_dir = SOURCE_ROOT / "images"
    labels_dir = SOURCE_ROOT / "labels"

    images = sorted(
        p for p in images_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

    assert len(images) >= 10, (
        "Недостаточно изображений для разделения 70/15/15"
    )

    for image in images:
        relative = image.relative_to(images_dir).with_suffix(".txt")
        label = labels_dir / relative
        assert label.is_file(), f"Не найдена разметка: {label}"

    random.Random(42).shuffle(images)

    n_train = int(len(images) * 0.70)
    n_val = int(len(images) * 0.15)

    subsets = {
        "train": images[:n_train],
        "val": images[n_train:n_train + n_val],
        "test": images[n_train + n_val:],
    }

    ROOT = EXTRACT_DIR / "prepared_dataset"

    for split, pictures in subsets.items():
        for image in pictures:
            relative = image.relative_to(images_dir)

            image_target = ROOT / "images" / split / relative
            label_target = (
                ROOT / "labels" / split / relative.with_suffix(".txt")
            )
            label_source = labels_dir / relative.with_suffix(".txt")

            image_target.parent.mkdir(parents=True, exist_ok=True)
            label_target.parent.mkdir(parents=True, exist_ok=True)

            shutil.copy2(image, image_target)
            shutil.copy2(label_source, label_target)

        print(f"{split}: {len(pictures)} изображений")

print("Рабочий датасет:", ROOT)

In [ ]:
from collections import Counter
import math

available_splits = ["train", "val"]

if (ROOT / "images/test").is_dir():
    available_splits.append("test")

for split in available_splits:
    images_dir = ROOT / "images" / split
    labels_dir = ROOT / "labels" / split

    images = sorted(
        p for p in images_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

    assert images, f"Нет изображений: {images_dir}"
    assert labels_dir.is_dir(), f"Нет папки разметки: {labels_dir}"

    counts = Counter()
    missing = []

    for image in images:
        relative = image.relative_to(images_dir).with_suffix(".txt")
        label = labels_dir / relative

        if not label.is_file():
            missing.append(str(relative))
            continue

        for line_number, line in enumerate(
            label.read_text(encoding="utf-8-sig").splitlines(), 1
        ):
            if not line.strip():
                continue

            try:
                values = list(map(float, line.split()))
                assert len(values) == 5

                cls, x, y, w, h = values

                assert all(math.isfinite(v) for v in values)
                assert cls in (0, 1)
                assert 0 <= x <= 1 and 0 <= y <= 1
                assert 0 < w <= 1 and 0 < h <= 1

            except (ValueError, AssertionError):
                raise ValueError(
                    f"Некорректная разметка: "
                    f"{label}:{line_number}\n{line}"
                )

            counts[int(cls)] += 1

    assert not missing, (
        f"{split}: нет TXT для {len(missing)} изображений:\n"
        + "\n".join(missing[:10])
    )

    print(
        f"{split}: изображений={len(images)}, "
        f"box_hitech={counts[0]}, "
        f"box_hitech_defect={counts[1]}"
    )

    assert counts[0] > 0 and counts[1] > 0, (
        f"В выборке {split} нужны размеченные примеры обоих классов."
    )

config = {
    "path": str(ROOT),
    "train": "images/train",
    "val": "images/val",
    "names": CLASS_NAMES,
}

if "test" in available_splits:
    config["test"] = "images/test"

DATA_YAML = ROOT / "data.yaml"

DATA_YAML.write_text(
    yaml.safe_dump(config, allow_unicode=True, sort_keys=False),
    encoding="utf-8",
)

print("\nГотово. Конфигурация для обучения:")
print(DATA_YAML.read_text())

In [ ]:
from ultralytics import YOLO
from pathlib import Path
from datetime import datetime
import shutil

assert DATA_YAML.is_file(), f"Не найден файл: {DATA_YAML}"

model = YOLO("yolo11n.pt")

model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=4,
    device="cpu",
    workers=2,
    patience=10,
    seed=42,
    plots=True,
    project="/content/drive/MyDrive/yolo_boxes",
    name="two_classes_cpu_"
         + datetime.now().strftime("%Y%m%d_%H%M%S"),
)

RUN_DIR = Path(model.trainer.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights/best.pt"

shutil.copy2(DATA_YAML, RUN_DIR / "dataset_config.yaml")

print("Лучшие веса:", BEST_WEIGHTS)
print("Все результаты:", RUN_DIR)

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import yaml
import pandas as pd
import shutil

RUN_DIR = Path(
    "/content/drive/MyDrive/yolo_boxes/"
    "two_classes_cpu_20261110_193341"
)

BEST_WEIGHTS = RUN_DIR / "weights/best.pt"
assert BEST_WEIGHTS.is_file(), f"Не найдены веса: {BEST_WEIGHTS}"

DATA_YAML = RUN_DIR / "dataset_config.yaml"
assert DATA_YAML.is_file(), f"Не найден YAML: {DATA_YAML}"

config = yaml.safe_load(DATA_YAML.read_text())
ROOT = Path(config["path"])

assert (ROOT / "images/test").is_dir(), (
    "Тестовые изображения отсутствуют. "
    "Если среда Colab сбрасывалась, нужно снова распаковать датасет "
    "и обновить путь path в YAML."
)

best_model = YOLO(str(BEST_WEIGHTS))

print("Модель:", BEST_WEIGHTS)
print("Классы:", best_model.names)

In [ ]:
metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=4,
    device="cpu",
    workers=2,
    plots=True,
    project=str(RUN_DIR),
    name="test_report",
)

REPORT_DIR = Path(metrics.save_dir)

rows = [{
    "Класс": "Все классы",
    "Precision": metrics.box.mp,
    "Recall": metrics.box.mr,
    "mAP50": metrics.box.map50,
    "mAP50–95": metrics.box.map,
}]

for index, class_id in enumerate(metrics.box.ap_class_index):
    precision, recall, ap50, ap5095 = metrics.box.class_result(index)

    rows.append({
        "Класс": best_model.names[int(class_id)],
        "Precision": precision,
        "Recall": recall,
        "mAP50": ap50,
        "mAP50–95": ap5095,
    })

table = pd.DataFrame(rows)

display(table.round(3))
table.to_csv(
    REPORT_DIR / "test_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Таблица сохранена:", REPORT_DIR / "test_metrics.csv")

In [ ]:
from IPython.display import display, Image


training_plot = RUN_DIR / "results.png"

if training_plot.is_file():
    print("Графики обучения")
    display(Image(filename=str(training_plot), width=1000))


for picture in sorted(REPORT_DIR.glob("*.png")):
    print(picture.name)
    display(Image(filename=str(picture), width=900))


history = pd.read_csv(RUN_DIR / "results.csv")
history.columns = history.columns.str.strip()

print("Последние 5 эпох:")
display(history.tail())

In [ ]:
for result in best_model.predict(
    source=str(ROOT / "images/test"),
    conf=0.25,
    imgsz=640,
    device="cpu",
    save=True,
    save_txt=True,
    save_conf=True,
    show_labels=True,
    show_conf=True,
    project=str(RUN_DIR),
    name="test_samples",
    exist_ok=True,
    stream=True,
):
    pass

SAMPLES_DIR = RUN_DIR / "test_samples"

pictures = sorted(
    p for p in SAMPLES_DIR.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
)

for picture in pictures[:10]:
    display(Image(filename=str(picture), width=900))

print("Все пробы сохранены:", SAMPLES_DIR)

In [ ]:
import pandas as pd

rows = []
image_rows = []

for result in best_model.predict(
    source=str(ROOT / "images/test"),
    conf=0.25,
    imgsz=640,
    device="cpu",
    stream=True,
    verbose=False,
):
    image_rows.append({
        "image": result.path,
        "detections": len(result.boxes),
    })

    for box in result.boxes:
        class_id = int(box.cls.item())
        confidence = float(box.conf.item())
        x1, y1, x2, y2 = box.xyxy[0].tolist()

        rows.append({
            "image": result.path,
            "class_id": class_id,
            "class_name": result.names[class_id],
            "confidence": confidence,
            "confidence_percent": confidence * 100,
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
        })

detections = pd.DataFrame(rows, columns=[
    "image", "class_id", "class_name",
    "confidence", "confidence_percent",
    "x1", "y1", "x2", "y2",
])

image_summary = pd.DataFrame(image_rows)

detections.to_csv(
    RUN_DIR / "detections_confidence.csv",
    index=False,
    encoding="utf-8-sig",
)

image_summary.to_csv(
    RUN_DIR / "images_detection_counts.csv",
    index=False,
    encoding="utf-8-sig",
)

display(detections.round(3))
print("Изображений без детекций:",
      int((image_summary["detections"] == 0).sum()))